# Week 7 · Wednesday: Functional API, Layers, Batch Norm, Saving and Two Full Builds

On Monday the digit network was a straight stack of layers, and yesterday that stack, `Sequential`, was the topic. Today:

1. **Functional composition:** layers called like functions, and a shape a straight stack cannot build
2. **Predefined layers:** the ready-made layers, chosen by what they read and what they hand on
3. **Batch normalization:** a scaler inside the network, measured like any change
4. **Build: a CNN digit classifier:** Monday's problem again, with convolutions, early stopping and dropout
5. **Saving and loading a model:** a `.keras` file, and the same predictions back from it
6. **Build: classifying penguins from a table:** blank cells, category columns and scaling, all learned from the training rows
7. **Variant: predicting a number:** the same table turned into a regression

Both builds follow the order your own project notebook will: problem statement, dataset, split, preprocessing, model, training, evaluation, error analysis, one improvement attempt, the test set once, limitations.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

Run the next cell first. It holds the imports and the two plotting helpers we reuse all day, and it fixes the random seeds so your numbers match the ones on the screen.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow together


def plot_history(history, metric="loss", title=""):
    """Draw the training and validation curves of one metric against the epoch, on one chart."""
    epochs = range(1, len(history.history[metric]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history[metric], marker="o", markersize=4, linewidth=2, label=f"training {metric}")
    plt.plot(epochs, history.history["val_" + metric], marker="o", markersize=4, linewidth=2,
             label=f"validation {metric}")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def show_image_grid(images, titles, columns=5):
    """Draw images (784-long rows or 28 by 28 by 1 arrays) as 28 by 28 pictures, in rows of `columns`."""
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(2.2 * columns, 2.4 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image, title in zip(axes.ravel(), images, titles):
        ax.imshow(np.reshape(image, (28, 28)), cmap="gray")
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


print(f"TensorFlow {tf.__version__}, Keras {tf.keras.__version__}, pandas {pd.__version__}, "
      f"scikit-learn {sklearn.__version__}, seaborn {sns.__version__}")
print("Setup complete")

## 1. Functional composition

Every model so far has been a straight stack: `tf.keras.Sequential` takes a list of layers, and each layer feeds the next. Keras has a second way to build a model, the **Functional API**, and it can build shapes a straight stack cannot.

All morning we use Monday's MNIST split, prepared exactly as on Monday: each image flattened to 784 pixels and scaled to 0 to 1, the first 54,000 training images for training and the last 6,000 for validation. The 10,000 test images are put aside until a build's test cell.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0
x_tr, y_tr = x_train[:54000], y_train[:54000]        # training rows
x_val, y_val = x_train[54000:], y_train[54000:]      # Monday's 6,000 validation rows
print(f"x_tr {x_tr.shape}, x_val {x_val.shape}, x_test {x_test.shape} (put aside)")

### 1.1 Monday's digit network, the Functional way

The Functional API builds a model in three moves:

1. Make an input with `tf.keras.Input(shape=(784,))`.
2. Call each layer on the result of the one before, as if the layer were a function. `Dense(128, activation="relu")` makes the layer, and the `(inputs)` straight after it calls the layer on the input.
3. Hand the first and last results to `tf.keras.Model(inputs, outputs)`.

Each call returns a **Keras tensor**: a placeholder that records a shape and the layers that led to it, and holds no numbers yet. The calls draw a map of the network, and real data flows along it later.

In [ ]:
inputs = tf.keras.Input(shape=(784,))                          # 1. the input: 784 pixels per row
x = tf.keras.layers.Dense(128, activation="relu")(inputs)      # 2. a layer, called on the input
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)   #    the next layer, called on its result
functional_model = tf.keras.Model(inputs, outputs)             # 3. wrap the first and last tensors

print(f"x.shape {x.shape}, outputs.shape {outputs.shape}")
print(f"Two rows of zeros through the model: shape {functional_model(np.zeros((2, 784), dtype='float32')).shape}")
functional_model.summary()

Monday built the same network as a Sequential list. Here it is again, beside the Functional one:

In [ ]:
sequential_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
print(f"Sequential: {sequential_model.count_params():,} parameters")
print(f"Functional: {functional_model.count_params():,} parameters")

- `x.shape` is `(None, 128)`. `None` is the batch size, left open until data arrives. Two real rows through the finished model come back as shape `(2, 10)`: one row of ten probabilities each.
- Both builds hold 101,770 parameters: 784 × 128 + 128 = 100,480 in the hidden layer and 128 × 10 + 10 = 1,290 in the output. The Functional summary also lists the input as a row of its own, with 0 parameters.
- Same layers, same count: the Functional model is Monday's network written a second way, and it compiles, fits and predicts the same way. A straight stack with one input and one output stays shorter as `Sequential`.
- An older library you may meet in tutorials, TFLearn, called these pieces `input_data`, `fully_connected` and a stack of layers. Today they are `tf.keras.Input`, `Dense` and `Sequential`. Section 5 says more about TFLearn.
- Try changing `Dense(128, ...)` to `Dense(64, ...)` in the Functional cell and rerun it: 50,890 parameters (784 × 64 + 64, plus 64 × 10 + 10). Put it back to 128.
- A common slip: `x = tf.keras.layers.Dense(128, activation="relu")` without the `(inputs)`. Then `x` is a layer that was never called, and the model will not build from it.

**In an interview:** "The Functional API builds a model by calling layers on tensors like functions and wrapping the input and the output in `tf.keras.Model`. I reach for it when the model branches, merges or has several inputs or outputs; a straight stack stays Sequential."

### 1.2 A shape Sequential cannot build

A **skip connection** carries a layer's input past one or more layers and joins it to a later layer's output. Here the raw 784 pixels skip past the hidden layer and join its 128 outputs, so the output layer reads both the image and what the hidden layer made of it.

`tf.keras.layers.Concatenate` does the join: it lays a list of tensors side by side along the last axis. `inputs` is now used twice, by the hidden layer and by the join. A Sequential model hands each layer's output only to the next layer, so this model needs the Functional API.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)     # the same starting weights on every rerun

inputs = tf.keras.Input(shape=(784,))
hidden = tf.keras.layers.Dense(128, activation="relu")(inputs)
joined = tf.keras.layers.Concatenate()([inputs, hidden])            # the skip: pixels beside the hidden output
outputs = tf.keras.layers.Dense(10, activation="softmax")(joined)   # Try changing it: (hidden) instead of (joined)
skip_model = tf.keras.Model(inputs, outputs, name="skip_model")

print(f"joined.shape {joined.shape}")
skip_model.summary()

- `joined` holds 784 + 128 = 912 values per row, so the output layer holds 912 × 10 + 10 = 9,130 parameters instead of 1,290. With the hidden layer's 100,480 the model holds 109,610: 7,840 more than Monday's, one weight from every pixel to every output unit.
- Try changing the output line to call the layer on `hidden` instead of `joined`, and rerun this cell only: 101,770 again, and the `Concatenate` row is gone. Put it back and rerun, because 1.3 draws this model.

The skip model compiles and trains like any other. Three epochs, `batch_size=128`, and Monday's validation rows:

In [ ]:
skip_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
skip_history = skip_model.fit(x_tr, y_tr, epochs=3, batch_size=128,
                              validation_data=(x_val, y_val), verbose=0)

for epoch, val_accuracy in enumerate(skip_history.history["val_accuracy"], start=1):
    print(f"epoch {epoch}: validation accuracy {val_accuracy:.4f}")

- The skip model trains like any other: validation accuracy climbs each epoch and ends at about 96.8% after 3 epochs.
- Read that as a sign the shape works. It settles nothing about whether the skip helps: Monday trained for 10 epochs, and judging the skip would need both models trained for the same epochs on the same validation rows, the way section 3 compares two models.
- The other join you meet in other people's code is `Add()([a, b])`, which adds two tensors value by value instead of placing them side by side. TFLearn's `merge` is `Concatenate` or `Add` today.

**In an interview:** "A skip connection carries an earlier tensor past some layers and joins it to a later one with `Concatenate` or `Add`. Sequential passes each output only to the next layer, so a skip needs the Functional API."

### 1.3 The branch as a picture

*Going deeper.* The idea to keep: two arrows leave the skip model's input, and that split is what a Sequential stack cannot draw.

`tf.keras.utils.plot_model` draws a model's layers as boxes joined by arrows, and `show_shapes=True` writes each layer's shapes on its box. It needs two drawing tools, the `pydot` package and the Graphviz program. Colab ships both, so there is nothing to install. On a laptop without them it raises an error, and that error says nothing about your model.

In [ ]:
tf.keras.utils.plot_model(skip_model, show_shapes=True)      # Try changing it: show_shapes=False

- Two arrows leave the input box: one to the hidden `Dense`, one to `Concatenate`.
- `summary()` carries the same facts as text. If the picture fails on another machine, move on with the summary.

## 2. Predefined layers

Keras ships ready-made layers in `tf.keras.layers`, and a team rarely writes its own. Choosing one starts from a plain question: what shape does this layer read, and what shape does it hand on?

### 2.1 The layers a team reaches for

Each row describes one example; the batch axis in front is left out.

| Layer | Reads | Hands on | Today |
|---|---|---|---|
| `Dense(units)` | a row of numbers | a row of `units` numbers | every model |
| `Flatten()` | several axes, such as a 28 by 28 image | one long row (784) | the CNN build |
| `Dropout(rate)` | any values | the same shape, a random share set to 0 while training | 4.5 |
| `BatchNormalization()` | any values | the same shape, rescaled per batch | section 3 |
| `Conv2D(filters, size)` | an image with a channel axis | a stack of feature maps | 2.3, the CNN build |
| `MaxPooling2D()` | feature maps | the same maps at half the height and width | 2.3, the CNN build |
| `Embedding(input_dim, output_dim)` | whole-number token ids | one vector per id | 2.4 |

Two words from the table:

- The **channel axis** is the last axis of an image array: one number per pixel for each colour. A grey digit has one channel.
- A **feature map** is a convolution's output grid: one number for each position its filter visited.

`Dropout` and `BatchNormalization` hand on exactly the shape they read. `Flatten`, `Dropout` and `MaxPooling2D` have no weights of their own: they reshape, switch off or shrink values.

**In an interview:** "I choose a layer by what it reads and what it hands on: `Dense` for a row of numbers, `Conv2D` and `MaxPooling2D` for images with a channel axis, `Embedding` for token ids, and `Flatten` to turn an image into a row for `Dense`."

### 2.2 Shapes in and out

A layer called on a real array computes numbers straight away (a call on an `Input` only draws the map). Here 32 random grey "images" stand in for MNIST: `np.random.rand` gives values between 0 and 1, and `astype("float32")` gives the 32-bit floats a network reads.

In [ ]:
images = np.random.rand(32, 28, 28).astype("float32")    # Try changing it: 8 images instead of 32
flat = tf.keras.layers.Flatten()(images)
dense_out = tf.keras.layers.Dense(128)(flat)             # Try changing it: Dense(10)
print(f"images {images.shape} -> Flatten {flat.shape} -> Dense(128) {dense_out.shape}")

- `Flatten` keeps the first axis, the batch of 32, and joins the rest into one row of 28 × 28 = 784. `Dense(128)` then hands on 128 numbers per image.
- Try changing the batch to 8: `(8, 28, 28)`, `(8, 784)`, `(8, 128)`. The batch axis passes through every layer untouched. Try `Dense(10)`: the last shape becomes `(32, 10)`. Put both back.

When a layer gets a shape it cannot read, Keras raises an error that names the layer and the shape it was given. Here the 784-input model from 1.1 gets the images without `Flatten`:

In [ ]:
try:
    functional_model(images)                 # images are (32, 28, 28); the model expects (None, 784)
except Exception as err:
    print(f"{type(err).__name__}: {str(err)[:240]}")

- Read the error as "the layer in front handed on the wrong shape". The fix is usually a reshape or a `Flatten` in front of the layer it names.
- The first number of a shape is the batch, never a feature count.

**In an interview:** "To check that layers fit together I print the shape each one hands on, or read `model.summary()`. A shape error names the layer that got the wrong shape, and the fix is usually a reshape in front of it."

### 2.3 Image layers

On week 6's Wednesday a **kernel**, a small grid of weights, slid across an image to make a feature map, and max pooling shrank the map. `Conv2D` is that operation as a layer that **learns** its kernels during training, and `MaxPooling2D` is the pooling step as a layer.

`Conv2D` reads a batch shaped (batch, height, width, channels). Our grey images have no channel axis, so `images[..., np.newaxis]` adds one: the `...` keeps every existing axis, and `np.newaxis` puts an axis of length 1 after them.

In [ ]:
images_4d = images[..., np.newaxis]                  # (32, 28, 28) -> (32, 28, 28, 1)
conv = tf.keras.layers.Conv2D(16, 3)                 # 16 kernels, each 3 by 3. Try changing it: Conv2D(16, 5)
maps = conv(images_4d)
pooled = tf.keras.layers.MaxPooling2D()(maps)        # a 2 by 2 window that keeps the largest value
print(f"images {images_4d.shape} -> Conv2D {maps.shape} -> MaxPooling2D {pooled.shape}")
print(f"Conv2D parameters: {conv.count_params()}")

- By default (`padding="valid"`) a 3 by 3 kernel only sits where it fits inside the image: 28 − 3 + 1 = 26 positions along each side. Sixteen kernels make sixteen 26 by 26 maps: `(32, 26, 26, 16)`.
- 160 parameters: 3 × 3 × 1 × 16 = 144 weights, plus one bias per kernel. The same small kernels slide across the whole image, so the count does not grow with the image size.
- `MaxPooling2D()` keeps the largest value in each 2 by 2 window, so 26 by 26 becomes 13 by 13. It has no parameters.
- Try `Conv2D(16, 5)`: `(32, 24, 24, 16)`, then `(32, 12, 12, 16)` after pooling, and 416 parameters (5 × 5 × 1 × 16 + 16). Put it back.
- Feed `Conv2D` the images without the channel axis and it raises a shape error naming the layer: the fix is `images[..., np.newaxis]` or `reshape(-1, 28, 28, 1)`.

**In an interview:** "A `Conv2D` layer slides small learned kernels over an image with a channel axis; `Conv2D(16, 3)` on 28 by 28 by 1 gives sixteen 26 by 26 feature maps from 160 parameters. A Dense layer on a flattened image ignores which pixels are neighbours; a convolution keeps that, with far fewer weights."

### 2.4 A text layer

*Going deeper.* The idea to keep: an `Embedding` layer takes whole-number token ids and hands on one vector per id, so (batch, length) becomes (batch, length, output_dim).

A sentence reaches a network as a list of **token ids**: one whole number per word, looked up in a vocabulary (week 6's Wednesday built one). Here are two made-up sentences of five ids each, from a vocabulary of 1,000 words. What the vectors mean is week 8's topic, and Friday's page uses this layer; today we read the shape only.

In [ ]:
token_ids = np.array([[4, 25, 7, 0, 0],
                      [12, 3, 999, 41, 6]])               # 2 sentences, 5 token ids each
embedding = tf.keras.layers.Embedding(input_dim=1000, output_dim=8)   # Try changing it: output_dim=16
print(f"token ids {token_ids.shape} -> Embedding {embedding(token_ids).shape}")

- Each of the 10 ids came back as a vector of 8 numbers: `(2, 5, 8)`. The last axis follows `output_dim`.

## 3. Batch normalization

### 3.1 The idea

Week 6's `StandardScaler` subtracts each column's mean and divides by its standard deviation (its spread), so every column sits around 0 with a spread of 1. **Batch normalization** is a layer that does the same rescaling inside the network, to the values a layer receives, and recomputes it on every training batch. It then multiplies by a learned scale and adds a learned offset, so training can undo the rescaling where that helps.

It behaves differently in two modes:

- **While training** (`fit`) it uses the current batch's mean and variance (variance is the spread squared).
- **While predicting** (`predict`, `evaluate`) there may be a single row with no spread of its own, so it uses **moving averages**: running averages of the mean and the variance, nudged towards each batch's values during training.

One column holding 2, 4 and 6, through a fresh layer in both modes:

In [ ]:
bn = tf.keras.layers.BatchNormalization()
batch = np.array([[2.0], [4.0], [6.0]], dtype="float32")      # Try changing it: 20, 40 and 60
print(f"prediction mode (training=False): {bn(batch, training=False).numpy().round(2).ravel()}")
print(f"training mode   (training=True):  {bn(batch, training=True).numpy().round(2).ravel()}")

- Training mode: the batch's mean is 4 and its variance (4 + 0 + 4) / 3 ≈ 2.67, a spread of about 1.63, so 2, 4 and 6 come out about −1.22, 0 and 1.22.
- Prediction mode: a fresh layer's moving averages start at mean 0 and variance 1, so the values pass through almost unchanged. After training they hold the averages the batches taught them.
- Try 20, 40 and 60: training mode prints the same −1.22, 0 and 1.22. The layer rescales whatever it is given. Put it back.
- After a 128-unit `Dense` layer it holds 4 × 128 = 512 numbers: 256 trainable (the scale and the offset) and 256 not trained by the optimizer (the moving mean and variance).
- It was introduced in 2015 (Ioffe and Szegedy) to reduce "internal covariate shift", the drift in a layer's inputs as the layers before it change, and so speed up training. Later work disputes that explanation, so we measure it in 3.2 instead of promising a gain. TFLearn's `batch_normalization` is `tf.keras.layers.BatchNormalization` today.

**In an interview:** "Batch normalization rescales a layer's inputs to about mean 0 and spread 1 using each training batch, then applies a learned scale and offset. At prediction time it uses moving averages kept during training, because one row has no spread of its own."

### 3.2 A worked improvement attempt

An **improvement attempt** is one change, measured before and after on the same validation data. Here the change is one `BatchNormalization` layer after the hidden `Dense` of the 784-128-10 network.

- `plain_model` is the network as it is; `bn_model` adds the one layer.
- Both train on `x_tr` for 5 epochs with `batch_size=128` and Adam, and both are judged on `validation_data=(x_val, y_val)`. Everything else is the same, so any difference comes from the one layer.

Those settings are `fit`'s knobs: `epochs` is how many passes over the training rows, `batch_size` how many rows per weight update, the optimizer the update rule, and `validation_data` what the model is judged on after each epoch.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)

plain_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
], name="plain_model")
bn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.BatchNormalization(),           # the one change. Try changing it: move it above the Dense
    tf.keras.layers.Dense(10, activation="softmax"),
], name="bn_model")

for model in (plain_model, bn_model):
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"plain_model {plain_model.count_params():,} parameters, bn_model {bn_model.count_params():,}")
bn_model.summary()

`bn_model` holds 101,770 + 512 = 102,282 parameters; the summary splits them into trainable and non-trainable. Now both fits, one after the other (about a minute in all):

In [ ]:
results = []
histories = {}
for model in (plain_model, bn_model):
    histories[model.name] = model.fit(x_tr, y_tr, epochs=5, batch_size=128,
                                      validation_data=(x_val, y_val), verbose=0)
    val_loss, val_accuracy = model.evaluate(x_val, y_val, verbose=0)
    results.append({"model": model.name, "val_loss": val_loss, "val_accuracy": val_accuracy})

plot_history(histories["plain_model"], title="plain_model: loss per epoch")
plot_history(histories["bn_model"], title="bn_model: loss per epoch")
pd.DataFrame(results).round(4)

- On our run `bn_model` came out a little ahead: validation loss about 0.082 against about 0.090, and validation accuracy about 97.5% against about 97.3%, about 9 images more right out of 6,000.
- One validation image is 1 / 6,000 ≈ 0.017 percentage points of accuracy, so a difference of a few hundredths of a percent is a handful of images and counts as noise. Here the loss gap is the clearer sign; the accuracy gap is small.
- The verdict rule: batch normalization helped if its validation loss is clearly lower; otherwise the simpler model stays. If your run shows the opposite, that is a valid result too, and you report it.
- Judge on validation loss, never on training accuracy. Compare the pair with each other only: Monday's `digit_model` trained for 10 epochs, this pair for 5.
- Try changing (before the fit cell): move the `BatchNormalization` line above the hidden `Dense`, so it rescales the 784 pixels, and rerun the build cell only: 4 × 784 = 3,136 batch-norm numbers and 104,906 parameters. Put it back and rerun the build cell before fitting.

**In an interview:** "To tell whether batch normalization helped, I train the model with and without it on the same data and judge both on the same validation loss. On a shallow digit network after 5 epochs it lowered validation loss a little, which is the kind of small, measured gain to report, never to promise."

## 4. Build: a CNN digit classifier

A **CNN** (convolutional neural network) is a network built from `Conv2D` layers. This build is Project 1's image option, done in the order your own project notebook follows: each subsection title names its step.

### 4.1 Problem statement and the data

**Problem statement.** Read a 28 by 28 grey image of a handwritten digit and say which of 0 to 9 it is: the job a form scanner at a bank or a mail sorter at a post office does. It is Monday's problem on purpose, so the CNN's numbers sit beside Monday's Dense network.

**The dataset and the split.** MNIST from `tf.keras.datasets`, prepared in section 1: 54,000 training images, 6,000 validation images and 10,000 test images, labelled 0 to 9. The test set is held out now and used once, at the end; every choice before then is made on the validation data.

**Preprocessing.** The pixels are already scaled to 0 to 1. `Conv2D` reads images with a channel axis, so each 784-long row folds back into 28 by 28 by 1. In `reshape(-1, 28, 28, 1)` the `-1` tells numpy to work that axis out from the others.

In [ ]:
x_tr_img = x_tr.reshape(-1, 28, 28, 1)
x_val_img = x_val.reshape(-1, 28, 28, 1)      # Try changing it: (-1, 28, 28), and read the printed shape
x_test_img = x_test.reshape(-1, 28, 28, 1)    # read once, in 4.6
print(f"x_tr_img {x_tr_img.shape}, x_val_img {x_val_img.shape}, x_test_img {x_test_img.shape}")
print(f"Training images per digit 0 to 9: {np.bincount(y_tr)}")
show_image_grid(x_tr_img[:10], [f"label {label}" for label in y_tr[:10]])

- Each of the three tables now holds 28 by 28 by 1 images, in the same order as the rows, so the validation images are Monday's.
- Every digit has between 4,872 training images (the fives) and 6,112 (the ones), so no digit is rare.
- Try `(-1, 28, 28)` on the `x_val_img` line: the shape prints without the last 1, which `Conv2D` refuses. Put it back and rerun before 4.2.

**In an interview:** "For a CNN on grey images I scale the pixels to 0 to 1, give each image its channel axis, 28 by 28 by 1, and keep a validation set apart from the training images. The test set is put aside before anything else and touched once at the end."

### 4.2 The model and a reason for each choice

`cnn_base` is a Sequential stack: two blocks of `Conv2D` and `MaxPooling2D`, then `Flatten`, then the ten-way softmax output. The function below builds and compiles it; its `dropout` argument is for the improvement attempt in 4.5, and it is 0 here.

In [ ]:
def make_cnn(dropout=0.0, name="cnn"):
    """Build and compile the digit CNN; dropout > 0 adds a Dropout layer after Flatten."""
    tf.keras.utils.set_random_seed(RANDOM_STATE)            # the same starting weights on every rebuild
    layers = [
        tf.keras.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),    # 16 learned 3 by 3 kernels
        tf.keras.layers.MaxPooling2D(),                      # halve the height and width
        tf.keras.layers.Conv2D(32, 3, activation="relu"),    # 32 kernels over the 16 maps
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),                           # 5 x 5 x 32 maps -> one row of 800
    ]
    if dropout > 0:
        layers.append(tf.keras.layers.Dropout(dropout))
    layers.append(tf.keras.layers.Dense(10, activation="softmax"))
    model = tf.keras.Sequential(layers, name=name)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


cnn_base = make_cnn(name="cnn_base")
cnn_base.summary()

The shapes walk 28 × 28 × 1 → 26 × 26 × 16 → 13 × 13 × 16 → 11 × 11 × 32 → 5 × 5 × 32 (half of 11, rounded down) → 800 → 10. The parameters are 160 + 4,640 (3 × 3 × 16 × 32 + 32) + 8,010 (800 × 10 + 10) = **12,810**, against 101,770 for Monday's Dense network.

| Choice | Here | Reason |
|---|---|---|
| Layers | two `Conv2D` + `MaxPooling2D` blocks, then `Flatten` | a convolution reads neighbouring pixels together, so a stroke is found wherever it sits; pooling shrinks the maps the next layer reads; `Flatten` hands the last maps to `Dense` as one row |
| Activations | ReLU in both convolutions | Monday's hidden-layer choice: cheap, and it keeps positive values |
| Output layer | `Dense(10, activation="softmax")` | one probability per digit, adding up to 1 |
| Loss | `sparse_categorical_crossentropy` | Monday's cross-entropy, for labels given as the integers 0 to 9 |
| Optimizer | Adam | the week's default |

- Try changing the first `Conv2D(16, 3, ...)` to `Conv2D(32, 3, ...)` and rerun this cell: 17,578 parameters (320 + 9,248 + 8,010). Put it back and rebuild before 4.3.

**In an interview:** "I used two small convolution and pooling blocks because convolutions read neighbouring pixels and share their kernels across the image, so the model has 12,810 parameters against 101,770 for a Dense network on the same digits. A softmax output gives one probability per class, trained with sparse categorical cross-entropy and Adam."

### 4.3 Training with early stopping

**Early stopping** ends training when the validation loss has not improved for a set number of epochs, the **patience**, and with `restore_best_weights=True` it puts back the weights of the best epoch. It is a **callback**: an object handed to `fit` in `callbacks=[...]` that `fit` consults after every epoch.

- `epochs=20` is a ceiling: training runs at most 20 epochs, and stops sooner when early stopping cuts in.
- `patience=2`: if validation loss is lowest at epoch 4 and epochs 5 and 6 do not beat it, training stops after epoch 6 and keeps epoch 4's weights.
- `batch_size=128`: a weight update after every 128 images, about 422 updates per epoch on 54,000 images.

We make `early_stop` once and pass it to both CNN fits. After a fit, `early_stop.stopped_epoch` holds the epoch it stopped at, counted from 0 as Python counts (0 means it never stopped early). This fit takes a minute or two on a CPU runtime: talk through the cell below while it runs.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history_base = cnn_base.fit(x_tr_img, y_tr, epochs=20, batch_size=128,           # at most 20 epochs
                            validation_data=(x_val_img, y_val), callbacks=[early_stop], verbose=0)

for epoch, (loss, val_loss) in enumerate(zip(history_base.history["loss"], history_base.history["val_loss"]), start=1):
    print(f"epoch {epoch}: training loss {loss:.4f}, validation loss {val_loss:.4f}")
best_epoch = int(np.argmin(history_base.history["val_loss"])) + 1
print(f"Epochs run: {len(history_base.history['loss'])} of 20 (early_stop.stopped_epoch = {early_stop.stopped_epoch})")
print(f"Lowest validation loss at epoch {best_epoch}: restore_best_weights put those weights back")
plot_history(history_base, title="cnn_base: loss per epoch")

- On our run the validation loss fell for 12 epochs, to about 0.041, then rose at epochs 13 and 14. Two epochs without a better validation loss is the patience used up, so early stopping ended training after epoch 14 of 20 (`stopped_epoch` 13, counted from 0) and put back epoch 12's weights.
- Over those last epochs the training loss kept falling. Training loss falling while validation loss turns up is Monday's overfitting pattern, and early stopping ended the fit two epochs after it began.
- For the first six epochs validation loss sits below training loss: the training loss of an epoch is averaged over its batches while the weights were still improving, and the validation loss is measured once, at the end of the epoch. Around epoch 7 the curves cross, and from there the gap widens as the network starts to fit the training images more closely than new ones.
- `len(history_base.history["loss"])` is how many epochs ran; the chart has one point per epoch.
- Rerunning the fit cell without rebuilding the model carries on from the trained weights. Rebuild with the 4.2 cell first.

**In an interview:** "I train against a validation set with an `EarlyStopping` callback on validation loss, patience 2 and `restore_best_weights=True`, so training stops once validation loss stops improving and the model keeps its best epoch's weights."

### 4.4 Evaluation and error analysis

`predict` returns ten probabilities per image, and `argmax(axis=1)` picks the most probable digit. The **confusion matrix** is a table of true classes against predicted classes: the diagonal counts the right answers, every other cell counts a kind of mistake. `ConfusionMatrixDisplay` draws it with its axes labelled.

The build keeps a results table from here on: one row per trained model, with the numbers from `evaluate` on the validation images.

In [ ]:
val_pred = cnn_base.predict(x_val_img, verbose=0).argmax(axis=1)    # the most probable digit per image
val_loss, val_accuracy = cnn_base.evaluate(x_val_img, y_val, verbose=0)

results = []
results.append({"model": "cnn_base", "val_loss": val_loss, "val_accuracy": val_accuracy})
print(f"cnn_base validation accuracy {val_accuracy:.4f}: {np.sum(val_pred != y_val)} of {len(y_val)} images wrong")

ConfusionMatrixDisplay.from_predictions(y_val, val_pred)
plt.title("cnn_base on the 6,000 validation images")
plt.show()

Now the mistakes themselves: the three most common confusions, then the first ten wrong validation digits.

In [ ]:
mistakes = confusion_matrix(y_val, val_pred)
np.fill_diagonal(mistakes, 0)                                       # keep only the off-diagonal cells
for cell in np.argsort(mistakes.ravel())[::-1][:3]:
    true_digit, predicted_digit = divmod(cell, 10)
    print(f"true {true_digit} predicted as {predicted_digit}: {mistakes[true_digit, predicted_digit]} times")

wrong = np.where(val_pred != y_val)[0][:10]                         # Try changing it: [:15]
show_image_grid(x_val_img[wrong], [f"true {y_val[i]}, predicted {val_pred[i]}" for i in wrong])

- `cnn_base` gets about 98.9% of the validation images right, 67 wrong out of 6,000 on our run. Monday's Dense network on the same 6,000 rows got about 97.6%.
- Read the matrix by its axis labels: the rows are the true digit and the columns the predicted one, so the largest off-diagonal cell names the pair the model mixes up most. On our run the top three were a 4 read as a 9 (7 times), a 9 read as a 4 (4) and a 5 read as a 3 (3): 4 and 9 get mixed up both ways, and with 67 mistakes in all no other pair stands out.
- Many of the wrong digits are ones a person would hesitate over too: a 4 written as one stroke like a 1, a 7 with an extra crossbar, a thin 9 with a tiny loop, a 1 with a base like a 2, a 2 drawn like a d. Your grid may show a slightly different set.
- Try `[:15]` for three rows of five. Put it back.

**In an interview:** "After the accuracy I read the confusion matrix for which classes get mixed up, then look at the wrong examples for what they share. That tells me what to try next and what to write under limitations."

### 4.5 One improvement attempt: dropout

**Dropout** switches off a random share of a layer's values at each training step, and scales the rest up to make up for them, so the network cannot lean on any one value. `Dropout(0.3)` after `Flatten` switches off about 240 of the 800 values on each step. It does nothing when predicting or evaluating. Dropout is a **regularisation** technique: a change made to reduce overfitting (week 8 covers others).

`cnn_model` is `cnn_base` plus that one layer, fitted the same way: at most 20 epochs, `batch_size=128`, the same validation images and the same early stopping. A minute or two again.

In [ ]:
cnn_model = make_cnn(dropout=0.3, name="cnn_model")       # Try changing it: dropout=0.5
history_dropout = cnn_model.fit(x_tr_img, y_tr, epochs=20, batch_size=128,
                                validation_data=(x_val_img, y_val), callbacks=[early_stop], verbose=0)

val_loss, val_accuracy = cnn_model.evaluate(x_val_img, y_val, verbose=0)
results.append({"model": "cnn_model", "val_loss": val_loss, "val_accuracy": val_accuracy})
best_epoch = int(np.argmin(history_dropout.history["val_loss"])) + 1
print(f"Epochs run: {len(history_dropout.history['loss'])} of 20 (early_stop.stopped_epoch = {early_stop.stopped_epoch})")
print(f"Lowest validation loss at epoch {best_epoch}: restore_best_weights put those weights back")
plot_history(history_dropout, title="cnn_model, with Dropout(0.3): loss per epoch")

How to tell whether it helped: compare the validation loss, and the gap between the two curves at each model's best epoch. Dropout lowers training accuracy on purpose, so training numbers say nothing about the verdict.

In [ ]:
for name, history in [("cnn_base", history_base), ("cnn_model", history_dropout)]:
    best = int(np.argmin(history.history["val_loss"]))
    loss, val_loss = history.history["loss"][best], history.history["val_loss"][best]
    print(f"{name}: best epoch {best + 1}, training loss {loss:.4f}, validation loss {val_loss:.4f}, "
          f"gap {val_loss - loss:+.4f}")

best_cnn = cnn_model if results[1]["val_loss"] < results[0]["val_loss"] else cnn_base   # lower validation loss
print(f"Kept: {best_cnn.name}")
pd.DataFrame(results).round(4)

- On our run dropout helped: validation loss about 0.033 against about 0.041, and validation accuracy about 99.1% against about 98.9% (about 11 more images right). `cnn_model` is kept as `best_cnn`.
- Early stopping fired on both fits: `cnn_model` stopped after epoch 12 and kept epoch 10's weights, `cnn_base` after epoch 14 with epoch 12's. Both stopped well before the cap of 20.
- `cnn_model`'s gap is negative: its training loss is measured with dropout switched on, so it reads higher than its validation loss, which runs with every value. A narrower gap, or a negative one, means the network leans less on the training images.
- A difference of a few images on 6,000 is noise; say so plainly when a change did not help, and keep the simpler model.
- Try changing `dropout=0.3` to `dropout=0.5` and rerun the fit cell and this one (a minute or two): about 400 of the 800 values are off on each step. Read the gap and the validation loss against the 0.3 run. Put it back to 0.3 and rerun both cells, so `best_cnn` is the planned model.

**In an interview:** "Dropout switches off a random share of values at each training step so the network cannot lean on any one of them, and it is off at prediction time. I judge it on validation loss and on the gap between the training and validation curves, never on training accuracy."

### 4.6 The test set, once, and limitations

`best_cnn` is the model the results table kept. This is the only cell of the build that reads the test images.

In [ ]:
test_loss, test_accuracy = best_cnn.evaluate(x_test_img, y_test, verbose=0)     # the test set, once
print(f"{best_cnn.name} test accuracy {test_accuracy:.4f} with {best_cnn.count_params():,} parameters")

- `cnn_model` scored about 99.0% on the 10,000 test images, close to its validation number. Monday's Dense network, with 101,770 parameters, scored about 97.4% on the same test images; the CNN does better with about an eighth of the parameters.
- If the test number comes out a little below the validation number, report it as it is. Going back to change the model after seeing it would turn the test set into a second validation set.

**Limitations.** The model has only seen centred, clean, white-on-black digits of one size. A photo of a digit on a form, a letter, or a digit written off centre is outside what it learned.

**In an interview** ("Walk me through a model you built"): "I built a small CNN to read handwritten digits. I put the 10,000 test images aside first and kept 6,000 training images for validation, reshaped the scaled pixels to 28 by 28 by 1, and used two convolution and pooling blocks with a softmax output, trained with early stopping on validation loss. I read the confusion matrix and the wrong digits, tried dropout as one change, kept the model with the lower validation loss, and scored the test set once at the end: about 99.0% accuracy with 12,810 parameters."

## 5. Saving and loading a model

### 5.1 Save, load, same predictions

A trained model lives in the notebook's memory until it is written to a file. Saving means the minutes of training never have to run again, and it is how a model is handed over.

- `best_cnn.save("digits_cnn.keras")` writes one file holding the architecture, the weights, the optimizer state and the compile settings. The `.keras` ending selects the Keras 3 format.
- `tf.keras.models.load_model("digits_cnn.keras")` rebuilds the model from the file.
- `np.allclose(a, b)` compares two arrays number by number, within a tiny tolerance, and returns one True or False for the lot.

In [ ]:
best_cnn.save("digits_cnn.keras")
loaded = tf.keras.models.load_model("digits_cnn.keras")

same = np.allclose(best_cnn.predict(x_val_img[:100], verbose=0), loaded.predict(x_val_img[:100], verbose=0))
print(f"Same predictions on 100 validation images (1,000 probabilities): {same}")

The file keeps the compile settings too, so a loaded model evaluates straight away, with no `compile` and no training. The next cell only needs the setup cell and the MNIST cell to have run, so it also works after a restart:

In [ ]:
loaded = tf.keras.models.load_model("digits_cnn.keras")
val_loss, val_accuracy = loaded.evaluate(x_val.reshape(-1, 28, 28, 1), y_val, verbose=0)
print(f"Loaded {loaded.name}: validation loss {val_loss:.4f}, validation accuracy {val_accuracy:.4f}")

- `np.allclose` printed True: the loaded model is the trained model.
- The loaded model's validation numbers are `best_cnn`'s row in the results table, `cnn_model`'s on our run: validation loss about 0.033, accuracy about 99.1%. The model's name came back from the file too.
- Try changing: choose Runtime > Restart session, which wipes every variable. Run only the setup cell, the MNIST cell and the cell above. No training runs, and the same validation accuracy prints. Then click 5.2's first cell and choose Runtime > Run before to bring everything else back (the two CNN fits take a few minutes).

**In an interview:** "I save a Keras model with `model.save` to a path ending `.keras`, which keeps the architecture, the weights, the optimizer state and the compile settings in one file, and load it with `tf.keras.models.load_model`. I check the loaded model gives the same predictions with `np.allclose`."

### 5.2 Where the file lives

Colab runs your code on a **runtime**: a temporary virtual machine. Files written there live on its disk and are deleted when the runtime is recycled, so a saved model missing the next morning usually went with the runtime. Open the Files panel (the folder icon on the left) to see `digits_cnn.keras`, and right-click it and choose Download, or copy it to Drive, to keep it.

In [ ]:
size_kb = os.path.getsize("digits_cnn.keras") / 1024
print(f"digits_cnn.keras: {size_kb:.0f} KB on this runtime's disk")
print(f".keras files here: {sorted(name for name in os.listdir() if name.endswith('.keras'))}")

- The file is small, about 180 KB: 12,810 parameters, Adam's bookkeeping and a description of the layers.
- The name matters. A name ending `.keras` gives the Keras 3 format. **`.h5`** is the older format, which Keras still reads and writes with a warning. **`model.export(path)`** writes a TensorFlow SavedModel for serving behind an endpoint, which is not reloaded as a Keras model (week 11). **`tf.keras.callbacks.ModelCheckpoint`** saves the model, or only its weights, during `fit`, optionally only the best epoch, so a long fit cut short does not lose everything.

TFLearn, an older high-level library for TensorFlow 1 that the curriculum names, saved and loaded its models like this. As checked on 2026-11-18 its last release is 0.5.0 (2020-11-11), the last commit on its `master` branch is 2020-11-30, and it does not import on Colab's TensorFlow 2.21, so this block is never run:

**TFLearn (history, do not run)**

```python
model = tflearn.DNN(net)
model.fit(X, Y)
model.save("digits.tflearn")
model.load("digits.tflearn")
```

`DNN.save` and `DNN.load` are `model.save("....keras")` and `tf.keras.models.load_model` today.

**In an interview:** "A file saved in Colab lives on the runtime's temporary disk and is gone when the runtime is recycled, so I download it or copy it to Drive. I save with a `.keras` name; `.h5` is legacy, and `model.export` is for serving."

## 6. Build: classifying penguins from a table

The second build is Project 1's tabular option: a table with blank cells and category columns, classified into three species. Same order as the CNN build, with the preprocessing steps a table needs.

### 6.1 Problem statement

Tell three penguin species, Adelie, Chinstrap and Gentoo, apart from field measurements: the length and depth of the bill, the flipper length, the body mass, the island and the sex. It matters to researchers counting colonies by species, who measure birds in the field and need the species recorded the same way every time. They would use the model to check or fill in the species of a measured bird.

A problem statement names three things: what is predicted, why it matters, and who would use the model. Your project notebook opens the same way.

**In an interview:** "I start with what the model predicts, why it matters and who would use it: here, the species of a penguin from field measurements, for researchers counting colonies."

### 6.2 The dataset

seaborn, a plotting library, also ships a few small example tables. `sns.load_dataset("penguins")` downloads one from the internet (Colab can reach it): the Palmer penguins, birds measured on three islands in Antarctica. If that download ever fails, the cell reads the same table from the palmerpenguins project's CSV instead and says so.

In [ ]:
try:
    penguins = sns.load_dataset("penguins")
    source = 'sns.load_dataset("penguins")'
except Exception:                                  # fallback: the same table from the palmerpenguins project
    url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
    penguins = pd.read_csv(url).drop(columns="year")
    penguins["sex"] = penguins["sex"].str.capitalize()        # "male" -> "Male", as seaborn has it
    source = "the palmerpenguins CSV"

print(f"Loaded from {source}: {penguins.shape[0]} rows, {penguins.shape[1]} columns")
print("Blank cells per column:")
print(penguins.isna().sum())

In [ ]:
print(penguins.head())
print()
print(f"Islands: {sorted(penguins['island'].unique())}")
print(f"Sex values: {sorted(penguins['sex'].dropna().unique())}")
print(f"Birds per species: {penguins['species'].value_counts().to_dict()}")

- 344 birds in 7 columns. Blank cells: 2 in each of the four measurements (the same two birds, one of them row 3 above, have no measurements at all) and 11 in `sex`.
- 152 Adelie, 124 Gentoo and 68 Chinstrap: the classes are uneven, so the split below is stratified. The islands are Biscoe, Dream and Torgersen.
- `island` and `sex` hold words, so they need encoding before a network can read them. `species` is the label, so it never goes into the inputs.
- A network cannot read a blank. The blanks get filled, and the fill values must come from the training rows only, so the split comes first.

**In an interview:** "Before modelling I look at the table's size, its columns, the blank cells per column, the category columns and the count per class. Here: 344 birds, blanks in five columns, two category columns and three species of uneven size."

### 6.3 Split first

Every fill, encoding and scale below is learned from the training rows, so the training rows are set apart before anything is learned.

- The label is a number: `y = penguins["species"].map(SPECIES.index)` turns each name into its position in `SPECIES`, so Adelie is 0, Chinstrap 1 and Gentoo 2.
- The first `train_test_split` puts 20% aside as the test set; the second takes 25% of the rest for validation. `stratify` keeps each species' share the same in every part.

The test set is held out now and used once, at the end; every choice before then is made on the validation data.

In [ ]:
NUMERIC = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORIES = ["island", "sex"]
SPECIES = ["Adelie", "Chinstrap", "Gentoo"]
X = penguins[NUMERIC + CATEGORIES]                   # the inputs: no species column
y = penguins["species"].map(SPECIES.index)           # the label: 0, 1 or 2

X_rest, X_test, species_rest, species_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)            # Try changing it: test_size=0.3
X_tr, X_val, species_tr, species_val = train_test_split(
    X_rest, species_rest, test_size=0.25, stratify=species_rest, random_state=RANDOM_STATE)

print(f"Training {len(X_tr)}, validation {len(X_val)}, test {len(X_test)} birds")
print(f"Training birds per species: {np.bincount(species_tr)}")

- 344 birds: 69 to the test set, then a quarter of the remaining 275 (69) to validation, which leaves 206 for training.
- Try `test_size=0.3` in the first split: 180 training, 60 validation and 104 test. Put it back and rerun before 6.4.

**In an interview:** "I split before any preprocessing: a stratified test set first, then a validation set from the rest, so nothing I learn from the data has seen the validation or test rows."

### 6.4 Preprocessing from the training rows only

**Data leakage** is letting information from the validation or test rows reach training, for example by fitting a scaler on all rows. The validation number then no longer measures unseen data. So each step below learns from the training rows only and applies what it learned to all three tables:

1. **Fill the blanks.** Each measurement gets the training rows' median, each category the training rows' most common value (`mode()` can return several equally common values, so the first is taken).
2. **One-hot encode.** **One-hot encoding** turns a category column into one 0-or-1 column per category, with `pd.get_dummies`.
3. **Line the columns up.** A table that lacks a category gets fewer columns from `get_dummies`. `.reindex(columns=train_columns, fill_value=0)` gives the validation and test tables the training columns, in the training order, with 0 in any that are missing.
4. **Scale.** `StandardScaler` fitted on the training rows, applied to all three.

Step 3 on a tiny example first: a training table with all three islands, and a validation table with only Dream birds.

In [ ]:
tiny_train = pd.get_dummies(pd.DataFrame({"island": ["Biscoe", "Dream", "Torgersen"]}), columns=["island"])
tiny_val = pd.get_dummies(pd.DataFrame({"island": ["Dream", "Dream"]}), columns=["island"])
print(f"Validation columns from get_dummies: {list(tiny_val.columns)}")
print(tiny_val.reindex(columns=tiny_train.columns, fill_value=0).astype(int))

Alone, the validation table got one column; after `reindex` it has the training's three, and each Dream bird reads 0, 1, 0. Now the four steps on the real tables:

In [ ]:
fills = {**X_tr[NUMERIC].median().to_dict(),                  # numbers: the training rows' median
         **X_tr[CATEGORIES].mode().iloc[0].to_dict()}          # categories: the training rows' most common value
print(f"Fill values, from the training rows: {fills}")


def prepare(df, columns=None):
    """Fill blanks with the training values, one-hot encode, and line the columns up with the training table."""
    df = pd.get_dummies(df.fillna(fills), columns=CATEGORIES)
    return df if columns is None else df.reindex(columns=columns, fill_value=0)


X_tr_enc = prepare(X_tr)
train_columns = X_tr_enc.columns
X_val_enc = prepare(X_val, train_columns)
X_test_enc = prepare(X_test, train_columns)
print(f"Columns: {list(train_columns)}")

scaler = StandardScaler().fit(X_tr_enc)        # Try changing it: .fit(pd.concat([X_tr_enc, X_val_enc, X_test_enc]))
X_tr_s, X_val_s, X_test_s = (scaler.transform(t).astype("float32") for t in (X_tr_enc, X_val_enc, X_test_enc))
print(f"Shapes: training {X_tr_s.shape}, validation {X_val_s.shape}, test {X_test_s.shape}")
print(f"Blanks left: {int(X_tr_enc.isna().sum().sum() + X_val_enc.isna().sum().sum() + X_test_enc.isna().sum().sum())}")

- The fill values come from the 206 training birds only: medians of 45.0 mm bill length, 17.4 mm bill depth, 197.0 mm flipper length and 4,050 g body mass, and Biscoe and Male as the most common island and sex.
- Nine columns: 4 measurements, 3 islands and 2 sexes, and no blanks left in any table.
- Try changing the scaler line to fit on all three tables together and rerun this cell. Then ask: why can the validation number no longer be trusted? The validation and test birds helped set the mean and the spread, so the validation score stops measuring unseen data, whether or not the number moves. Put the line back and rerun before 6.5.

**In an interview:** "I fill blanks with values worked out from the training rows, one-hot encode the categories and reindex the other tables to the training columns, and fit the scaler on the training rows only. Anything learned from the validation or test rows is leakage."

### 6.5 The model and a reason for each choice

`penguin_model` is a small Dense network: 9 inputs, one hidden layer of 8 ReLU units, and 3 softmax outputs. The function builds and compiles it with the number of hidden units as its argument, because the improvement attempt in 6.9 changes only that.

In [ ]:
def make_penguin_model(units):
    """Build and compile the penguin classifier with one hidden layer of `units` ReLU units."""
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(9,)),
        tf.keras.layers.Dense(units, activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax"),      # one probability per species
    ], name=f"penguins_{units}_units")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


penguin_model = make_penguin_model(8)                       # Try changing it: make_penguin_model(4)
penguin_model.summary()

| Choice | Here | Reason |
|---|---|---|
| Layers | one hidden `Dense(8)` | a small table (206 training rows) needs a small model; 107 parameters cannot simply memorise 206 rows |
| Activation | ReLU | the week's hidden-layer default |
| Output layer | `Dense(3, activation="softmax")` | three species, one probability each, adding up to 1 |
| Loss | `sparse_categorical_crossentropy` | the labels are the integers 0, 1 and 2 |
| Optimizer | Adam | the week's default |

- 9 × 8 + 8 = 80 parameters in the hidden layer and 8 × 3 + 3 = 27 in the output: 107 in all.
- Try `make_penguin_model(4)`: 55 parameters (40 + 15). Put it back and rerun.

**In an interview:** "For a three-class table I used one small hidden ReLU layer and a three-unit softmax output with sparse categorical cross-entropy, sized to the 206 training rows."

### 6.6 Training with both curves

50 epochs with `batch_size=16`: each epoch is only 13 batches (twelve of 16 and a last one of 14), so the whole fit takes seconds.

In [ ]:
history_penguins = penguin_model.fit(X_tr_s, species_tr, epochs=50, batch_size=16,
                                     validation_data=(X_val_s, species_val), verbose=0)
print(f"Epoch 50: training loss {history_penguins.history['loss'][-1]:.4f}, "
      f"validation loss {history_penguins.history['val_loss'][-1]:.4f}")
plot_history(history_penguins, title="penguin_model: loss per epoch")

- Both curves fall together and end close, at about 0.05 training and 0.06 validation loss on our run, with no sign of the validation curve turning up: no overfitting in 50 epochs.
- Refitting without rebuilding carries on from the trained weights: rerun the 6.5 cell first.

**In an interview:** "I fit with a validation set and draw the training and validation loss on one chart; both falling together means it is learning, and validation rising while training falls would mean overfitting."

### 6.7 Evaluation on the validation data

`classification_report` prints, for each species:

- **precision**: of the birds called this species, the share that really are;
- **recall**: of the birds that really are this species, the share the model found;
- **F1**: precision and recall combined into one number, high only when both are;
- **support**: how many validation birds the species has.

Its last rows are two averages over the species, macro and weighted; week 8 covers them, so today we read the species rows.

In [ ]:
val_pred = penguin_model.predict(X_val_s, verbose=0).argmax(axis=1)
val_loss, val_accuracy = penguin_model.evaluate(X_val_s, species_val, verbose=0)

results = []
results.append({"model": "8 units", "val_loss": val_loss, "val_accuracy": val_accuracy})
print(f"Validation accuracy {val_accuracy:.4f}: {np.sum(val_pred != species_val)} of {len(species_val)} birds wrong")
print(classification_report(species_val, val_pred, target_names=SPECIES, digits=3))

ConfusionMatrixDisplay.from_predictions(species_val, val_pred, display_labels=SPECIES)
plt.title("penguin_model on the 69 validation birds")
plt.show()

- On our run one bird of 69 was wrong, a validation accuracy of about 98.6%: a Chinstrap called an Adelie. That one bird is why Chinstrap's recall is about 0.92 (12 of 13 found) and Adelie's precision about 0.97; every other species score is 1.
- The species rows say whether one species is being missed even when overall accuracy is high. Chinstrap has the fewest birds, so one mistake moves its precision and recall the most.

**In an interview:** "For a multi-class table I report accuracy, then precision, recall and F1 per class from `classification_report`, and the confusion matrix, because overall accuracy can hide a class that keeps being missed."

### 6.8 Error analysis

The validation birds the model gets wrong, printed with their original measurements so they can be read. Beside them, the five birds with the lowest `p_true`, the probability the model gave each bird's true species: the birds it was least sure about. That way there are rows to read even when every validation bird is right.

In [ ]:
proba = penguin_model.predict(X_val_s, verbose=0)
view = X_val.copy()                                          # the original, unscaled measurements
view["true"] = [SPECIES[i] for i in species_val]
view["predicted"] = [SPECIES[i] for i in val_pred]
view["p_true"] = proba[np.arange(len(proba)), species_val.to_numpy()].round(3)

print("Wrong validation rows:")
print(view[view["true"] != view["predicted"]])
print()
print("The five least sure rows:")
print(view.nsmallest(5, "p_true"))                           # Try changing it: nsmallest(10, ...)

To read them against the species, here are the training rows' average measurements per species:

In [ ]:
print(X_tr[NUMERIC].groupby([SPECIES[i] for i in species_tr]).mean().round(1))

- On our run the wrong bird is a Chinstrap from Dream called an Adelie, with `p_true` about 0.33. Its bill is 42.4 mm long, between the Adelie average of 39.0 and the Chinstrap average of 49.0, and its bill depth, flipper length and mass sit near both species' averages. It is a bird between two species, and bill length is what tells those two apart.
- Four of the five least sure birds are from Dream, the one island where both Adelies and Chinstraps live in this table, and each of those four has a bill length between the two species' averages.
- No Gentoo appears among the unsure rows: their shallow bills, long flippers and heavy bodies set them apart.

**In an interview:** "I print the rows the model gets wrong, and the ones it was least sure about, in their original units, and read them against each class's typical values. Rows that sit between two classes are expected mistakes; anything else points at the data or the model."

### 6.9 One improvement attempt

One change: the hidden layer widened from 8 to 32 units (9 × 32 + 32 = 320, plus 32 × 3 + 3 = 99, so 419 parameters). Same split, same preprocessing, same 50 epochs and batch size, same validation birds.

In [ ]:
wide_model = make_penguin_model(32)                          # the one change. Try changing it: 64
wide_model.fit(X_tr_s, species_tr, epochs=50, batch_size=16,
               validation_data=(X_val_s, species_val), verbose=0)
val_loss, val_accuracy = wide_model.evaluate(X_val_s, species_val, verbose=0)
results.append({"model": "32 units", "val_loss": val_loss, "val_accuracy": val_accuracy})

best_penguin_model = wide_model if results[1]["val_loss"] < results[0]["val_loss"] else penguin_model
print(f"{wide_model.count_params()} parameters; kept: {best_penguin_model.name}")
pd.DataFrame(results).round(4)

- On our run the 32-unit model got every validation bird right, with validation loss about 0.031 against about 0.060, so it is kept as `best_penguin_model`. It helped, by one bird and a clearly lower loss: it is surer of the right answers.
- One validation bird is 1 / 69 ≈ 1.4 percentage points of accuracy, so read accuracy differences in birds. The kept model is the row with the lower validation loss.
- Try 64 units: 835 parameters (640 + 195) and a new row to compare. Put it back to 32 and rerun.

**In an interview:** "I change one thing at a time, refit on the same split and compare validation loss and accuracy in a results table, then keep the better row. A bigger model is kept only if the validation numbers say so."

### 6.10 The test set, once, saving, and limitations

The kept model meets the 69 test birds once, and is saved to `penguins.keras`.

In [ ]:
test_loss, test_accuracy = best_penguin_model.evaluate(X_test_s, species_test, verbose=0)   # the test set, once
print(f"{best_penguin_model.name} test accuracy {test_accuracy:.4f} "
      f"({round((1 - test_accuracy) * len(species_test))} of {len(species_test)} birds wrong)")
best_penguin_model.save("penguins.keras")
print(f"Saved penguins.keras ({os.path.getsize('penguins.keras') / 1024:.0f} KB)")

- On our run the kept model got all 69 test birds right, a test accuracy of 1.0. Sixty-nine birds are too few to trust a perfect score: one more mistake would read about 0.986. It says these three species are easy to tell apart from these measurements, and it promises nothing about every new bird.
- Saving after the test cell is fine: the test cell changes nothing about the model.

**Limitations.** The model knows only 344 birds from three islands, and 69 test birds is a small test. A bird from a new island gets 0 in every island column, and a fourth species can only be called one of the three.

**In an interview** ("Walk me through a model you built"): "I classified three penguin species from a 344-row table. I split off a stratified test set first, then a validation set. I filled blanks with training medians and most common values, one-hot encoded island and sex, reindexed to the training columns and fitted the scaler on the training rows. A small Dense network with a softmax output trained for 50 epochs; I read the per-species report and the wrong and least sure rows, tried a wider hidden layer, kept the better model on validation loss, scored the test set once and saved it as a `.keras` file."

## 7. Variant: predicting a number

### 7.1 The same table, a numeric target

A **regression** predicts a number instead of a class. The same three splits now predict each bird's body mass, in kilograms, from the other three measurements and the categories `species`, `island` and `sex`.

- Rows with no `body_mass_g` are dropped. A row with no target is dropped, never filled: a filled target would teach the model an answer someone made up.
- The inputs are prepared as in section 6: the training fills, one-hot columns lined up with the training table's, a scaler fitted on the training rows. Species is now an input, so there are 11 columns: 3 measurements, 3 species, 3 islands and 2 sexes.

In [ ]:
MASS_CATEGORIES = ["species", "island", "sex"]
MASS_INPUTS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"] + MASS_CATEGORIES


def mass_table(X_part, species_part):
    """One split as a regression table: rows with no body mass dropped, the target in kilograms."""
    df = X_part.assign(species=[SPECIES[i] for i in species_part])
    df = df[df["body_mass_g"].notna()]                      # no target: drop the row, never fill it
    return df[MASS_INPUTS].fillna(fills), (df["body_mass_g"] / 1000).to_numpy("float32")


rows_tr, mass_tr = mass_table(X_tr, species_tr)
rows_val, mass_val = mass_table(X_val, species_val)
rows_test, mass_test = mass_table(X_test, species_test)

m_tr = pd.get_dummies(rows_tr, columns=MASS_CATEGORIES)
m_val = pd.get_dummies(rows_val, columns=MASS_CATEGORIES).reindex(columns=m_tr.columns, fill_value=0)
m_test = pd.get_dummies(rows_test, columns=MASS_CATEGORIES).reindex(columns=m_tr.columns, fill_value=0)
mass_scaler = StandardScaler().fit(m_tr)
M_tr, M_val, M_test = (mass_scaler.transform(t).astype("float32") for t in (m_tr, m_val, m_test))
print(f"Rows: training {len(mass_tr)}, validation {len(mass_val)}, test {len(mass_test)}; {M_tr.shape[1]} columns")

Three things change from the classifier:

- **The output layer** is `Dense(1)` with no activation, a **linear output**: it can produce any number, where a softmax or sigmoid would squeeze it.
- **The loss** is `"mse"`, Monday's mean squared error: the average squared miss.
- **The metric** is `"mae"`, **mean absolute error**: the average size of the miss, in the target's own unit (kilograms here).

`tf.keras.optimizers.Adam(learning_rate=0.01)` sets the optimizer's step size by hand, one more of `fit`'s knobs; 100 epochs with `batch_size=16` take a few seconds.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
mass_model = tf.keras.Sequential([
    tf.keras.Input(shape=(11,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),                          # linear output. Try changing it: activation="sigmoid"
], name="mass_model")
mass_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"])
history_mass = mass_model.fit(M_tr, mass_tr, epochs=100, batch_size=16,
                              validation_data=(M_val, mass_val), verbose=0)
print(f"{mass_model.count_params()} parameters")
plot_history(history_mass, metric="mae", title="mass_model: mean absolute error (kg) per epoch")

- 11 × 16 + 16 = 192 parameters in the hidden layer and 16 + 1 = 17 in the output: 209.
- Two training birds had no body mass, so 204 training rows remain; validation and test keep all 69.
- The chart draws the mean absolute error (`metric="mae"`), in kilograms: both curves drop fast in the first few epochs, then flatten out a little above 0.2 kg and stay close together.
- Try `Dense(1, activation="sigmoid")` and rerun: every prediction is squeezed below 1 kg while the birds weigh several kilograms, so the error jumps. Put it back and rerun.

**In an interview:** "To turn a classifier into a regression I change three things: the output becomes one unit with no activation, the loss becomes mean squared error, and I report mean absolute error, which is in the target's own unit."

### 7.2 Evaluating a regression

A regression is read with numbers and a plot: the validation MSE and MAE (the MAE also in grams), then predicted against actual mass. On that plot a perfect model puts every dot on the diagonal line, and a dot's distance from the line is that bird's error.

In [ ]:
val_mse, val_mae = mass_model.evaluate(M_val, mass_val, verbose=0)
print(f"Validation MSE {val_mse:.4f} kg squared, MAE {val_mae:.3f} kg (about {val_mae * 1000:.0f} g)")

pred = mass_model.predict(M_val, verbose=0)[:, 0]
plt.figure(figsize=(6, 6))
plt.scatter(mass_val, pred, s=40, edgecolor="black", alpha=0.8)
limits = [min(mass_val.min(), pred.min()) - 0.2, max(mass_val.max(), pred.max()) + 0.2]
plt.plot(limits, limits, "k--", label="perfect prediction")
plt.title("mass_model on the validation birds")
plt.xlabel("Actual body mass (kg)")
plt.ylabel("Predicted body mass (kg)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

The five validation birds with the largest errors: `np.argsort` orders the errors from smallest to largest, so its last five positions are the five largest misses.

In [ ]:
errors = pred - mass_val
largest = np.argsort(np.abs(errors))[-5:][::-1]             # Try changing it: [-10:]
print(rows_val.iloc[largest].assign(actual_kg=mass_val[largest], predicted_kg=pred[largest].round(2),
                                    error_kg=errors[largest].round(2)))

- Validation MAE about 0.23 kg, so the model is about 230 g off per bird on average (MSE about 0.088 kg squared). The dots on the plot scatter around the diagonal, most within about half a kilogram.
- On our run the largest miss is an Adelie of 2.975 kg, the lightest validation bird, predicted at about 3.9 kg; the next is an Adelie of 4.65 kg, also predicted at about 3.9. The misses are birds far from what birds like them usually weigh: the model predicts close to the typical mass for a bird's species, sex and size, so unusually light or heavy birds are where it goes wrong.
- Is that good enough? Ask the people using the model whether that many grams off per bird is close enough. The plot and the largest errors show where it goes wrong.

Last, the test set, once:

In [ ]:
test_mse, test_mae = mass_model.evaluate(M_test, mass_test, verbose=0)     # the test set, once
print(f"Test MAE {test_mae:.3f} kg (about {test_mae * 1000:.0f} g)")

- Test MAE about 0.21 kg, about 210 g per bird, close to the validation number.
- This variant carries no improvement attempt; the classification build did. A regression project would carry one too, measured on validation MAE.

**In an interview:** "For a regression I report MSE and MAE on the validation data, draw predicted against actual with the diagonal, read the rows with the largest errors, and score the test set once: here about 210 g off per bird."

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Functional API (`Input`, layer calls, `tf.keras.Model`) | way to build a Keras model | branches, merges, skip connections, several inputs or outputs | a straight stack is shorter as `Sequential`; a layer never called on a tensor builds nothing |
| Predefined layers | building blocks in `tf.keras.layers` | choosing by what a layer reads and hands on | `Conv2D` needs a channel axis; the first number of a shape is the batch |
| Batch normalization | layer | rescaling a layer's inputs per training batch | uses moving averages when predicting; measure it on the same validation rows |
| Early stopping | callback | stopping when validation loss stops improving | needs `restore_best_weights=True` to keep the best epoch |
| Dropout | layer, regularisation | reducing overfitting | lowers training accuracy on purpose; judge it on validation loss and the gap |
| CNN (`Conv2D`, `MaxPooling2D`, `Flatten`) | supervised, image classifier | images, with far fewer parameters than a Dense network | only knows images like its training images |
| `.keras` file | saving and loading | one file with architecture, weights, optimizer and compile settings | Colab's disk is wiped when the runtime is recycled; download it |
| Table preprocessing | split, fill, one-hot, scale | turning blanks and words into numbers a network reads | split first; learn every fill, column and scale from the training rows; `reindex` |
| Regression output | `Dense(1)`, `loss="mse"`, MAE | predicting a number | drop rows with no target; MAE is in the target's unit, MSE is squared |

**Quick check**

1. When would you use the Functional API instead of `Sequential`, and why can a skip connection not be built as a straight stack?
2. What shape does `Conv2D(16, 3)` hand on for a batch of 32 grey 28 by 28 images, and what has to happen to the images first?
3. What does a batch normalization layer do while training and while predicting, and how would you tell whether adding it helped?
4. Training loss keeps falling while validation loss rises. What is happening, and what do early stopping and dropout each do about it?
5. What is in a `.keras` file, and why might it be gone the next morning in Colab?
6. A teammate fills the blanks and fits the scaler on all 344 penguins, then splits. What is wrong, and how do you fix it?